# 03 - Baselines

Computes the three Model 0 baselines (monthly climatology, persistence,
linear interpolation) against the released chlorophyll artificial-gap pool
and scores them. Fully executable on the public data included in this
repository.

**Scoring scale.** The final report and `results_public/chlorophyll/chlorophyll_benchmark_summary.csv`
score chlorophyll on `log10(chl_mean)`, not on physical `chl_mean`
(`mg m^-3`), because the target distribution is strongly right-skewed. This
notebook reproduces that scale: it derives a `chl_log10` column from the
public daily target table and scores against `chl_log10`, so the MAE values
below are directly comparable to the released benchmark table. An earlier
version of this notebook scored directly on physical `chl_mean`, which is
*not* comparable to the log10-scale benchmark numbers — a physical-scale MAE
is dominated by a handful of high-chlorophyll days and is roughly an order
of magnitude larger. See `docs/methodology/target_and_gap_construction.md`
for why the log10 transform was chosen.


In [ ]:
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
from coastal_gap_reconstruction.artificial_gap_validation import apply_artificial_gap
from coastal_gap_reconstruction.baseline_imputation import run_all_baselines
from coastal_gap_reconstruction.data_loading import load_daily_target, load_validation_gap_pool
from coastal_gap_reconstruction.scoring_metrics import aggregate_metrics, compute_gap_metrics

target_df = load_daily_target("../data_public/chlorophyll/chlorophyll_daily_target.csv")
gap_pool = load_validation_gap_pool("../data_public/chlorophyll/chlorophyll_validation_gaps.csv")

# Score on log10(chl_mean), matching the benchmark scale (see the notebook
# intro and docs/methodology/target_and_gap_construction.md). chl_mean is
# guaranteed positive for eligible days; non-positive/NaN values map to NaN
# and are excluded downstream the same way missing values already are.
target_df["chl_log10"] = np.log10(target_df["chl_mean"].where(target_df["chl_mean"] > 0))
TARGET_COL = "chl_log10"

print(len(gap_pool), "gaps loaded")


## Run baselines on every gap in the pool

In [ ]:
all_metrics = []

for _, g in gap_pool.iterrows():
    start = pd.Timestamp(g["start_date"])
    gap_length = int(g["gap_length"])

    masked = apply_artificial_gap(target_df, start, gap_length, target_col=TARGET_COL)
    predictions = run_all_baselines(masked, start, gap_length, target_col=TARGET_COL)

    metrics = compute_gap_metrics(
        target_df=target_df,
        predictions=predictions,
        start_date=start,
        gap_length=gap_length,
        gap_id=g["gap_id"],
        gap_info=g.to_dict(),
        target_col=TARGET_COL,
    )
    all_metrics.extend(metrics)

metrics_df = pd.DataFrame(all_metrics)
metrics_df.head()


## Aggregate by method and gap length

In [ ]:
summary = aggregate_metrics(metrics_df, groupby_cols=["method", "gap_length"])
summary.sort_values(["method", "gap_length"])


## Aggregate by method only

In [ ]:
overall = aggregate_metrics(metrics_df, groupby_cols=["method"])
overall


## Interpretation

These three baselines establish the floor for this benchmark. Linear
interpolation typically performs best among the three for short gaps (it
has access to both edges of the gap), but is not forecast-safe. Because
these numbers are scored on `log10(chl_mean)`, they are directly comparable
to `results_public/chlorophyll/chlorophyll_benchmark_summary.csv`, which
scores the engineered tabular, gap-edge, and TS-ICL methods the same way.
Exact values will differ slightly from the released table because the
released benchmark applies additional bootstrap/day-weighting and stratum
matching described in the report; this notebook reproduces the underlying
per-gap MAE, not the exact published aggregation.
